# PRAGMA Phase 3 - Reference dataset, collator, and `PragmaBatch`

Loads tokenized shards through `TokenizedRecordDataset`, packs them into `PragmaBatch` objects with `PragmaCollator` (section 9.1; ADR 0004), and checks the Phase 3 exit gate directly: **batches reconstruct their source records exactly**, and **cross-event / cross-record boundaries are validated**.

This is the *reference* path: a fixed `batch_size` via a standard `torch.utils.data.DataLoader`. The dynamic `TokenBudgetBatchSampler` that groups records by token/event budget instead is a Phase 8 concern — this notebook is the baseline it will later be compared against.

No masking is applied yet (Phase 4): `event_mlm_labels` is always `-100` everywhere in every batch produced here.

In [1]:
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader

from pragma.data import (
    IGNORE_INDEX,
    ParquetShardStore,
    PragmaCollator,
    TokenizedRecordDataset,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SHARDS_DIR = REPO_ROOT / "data" / "shards"

train_dataset = TokenizedRecordDataset.from_store(
    SHARDS_DIR, ParquetShardStore(), split="train"
)
print(f"{len(train_dataset)} train records")

399 train records


## Build one reference batch

A standard `DataLoader` with a fixed batch size and `PragmaCollator` as `collate_fn` — no dynamic token-budget logic yet.

In [2]:
loader = DataLoader(
    train_dataset, batch_size=16, shuffle=True, collate_fn=PragmaCollator()
)
batch = next(iter(loader))

print(f"n_records: {batch.n_records}")
print(f"n_events: {batch.n_events}")
print(f"n_profile_tokens: {batch.n_profile_tokens}")
print(f"n_event_tokens: {batch.n_event_tokens}")

n_records: 16
n_events: 5638
n_profile_tokens: 160
n_event_tokens: 29618


## `validate()`: the boundary-safety check

Asserts `event_cu_seqlens`/`history_cu_seqlens` are non-decreasing, span exactly the token/event counts they should, and that `event_to_record` agrees with the grouping `history_cu_seqlens` implies. `PragmaCollator` already calls this before returning a batch — running it again here just makes the guarantee visible.

In [3]:
batch.validate()
print("boundaries validated: OK")

boundaries validated: OK


## Exit-gate check: batches reconstruct their source records exactly

For every record in the batch, slice its event value-token ids back out of the packed buffers using `event_to_record` and `event_cu_seqlens`, and compare against the source `TokenizedRecord`. This is the same check `tests/unit/test_batch.py::test_batch_reconstructs_source_records_exactly` runs on every commit, made visible here.

In [4]:
def reconstruct_event_value_ids(batch, record_idx: int) -> list[int]:
    event_indices = (batch.event_to_record == record_idx).nonzero(as_tuple=True)[0]
    ids: list[int] = []
    for event_idx in event_indices.tolist():
        start = int(batch.event_cu_seqlens[event_idx])
        end = int(batch.event_cu_seqlens[event_idx + 1])
        ids.extend(batch.event_value_ids[start:end].tolist())
    return ids


source_records = [train_dataset[i] for i in range(len(train_dataset))]
records_by_entity = {r.entity_id: r for r in source_records}

mismatches = []
for record_idx, entity_id in enumerate(batch.entity_ids):
    source = records_by_entity[entity_id]
    expected = [tid for e in source.events for tid in e.value_ids()]
    actual = reconstruct_event_value_ids(batch, record_idx)
    if actual != expected:
        mismatches.append(entity_id)

print(f"mismatched records: {len(mismatches)} / {batch.n_records}")
assert not mismatches

mismatched records: 0 / 16


## Exit-gate check: cross-event and cross-record boundaries hold

Every event's tokens fall strictly within its own `event_cu_seqlens` slice, and every event assigned to one record's `history_cu_seqlens` slice actually points back to that record in `event_to_record` — the two structural guarantees a future attention backend needs to never let one event or one customer's history leak into another's.

In [5]:
event_idx = 0
event_boundary_ok = True
for record_idx, entity_id in enumerate(batch.entity_ids):
    source = records_by_entity[entity_id]
    for event in source.events:
        start = int(batch.event_cu_seqlens[event_idx])
        end = int(batch.event_cu_seqlens[event_idx + 1])
        if batch.event_key_ids[start:end].tolist() != event.key_ids():
            event_boundary_ok = False
        event_idx += 1

history_boundary_ok = True
for record_idx in range(batch.n_records):
    start = int(batch.history_cu_seqlens[record_idx])
    end = int(batch.history_cu_seqlens[record_idx + 1])
    if not bool((batch.event_to_record[start:end] == record_idx).all()):
        history_boundary_ok = False

print(f"event boundaries hold: {event_boundary_ok}")
print(f"history boundaries hold: {history_boundary_ok}")
assert event_boundary_ok and history_boundary_ok

event boundaries hold: True
history boundaries hold: True


## Zero-event entities need no special-casing

ADR 0004 calls this out explicitly: the synthetic corpus's zero-event cohort is a required fixture for testing that `PragmaBatch` handles an empty event history without a special code path. Confirm one directly.

In [6]:
zero_event_records = [r for r in source_records if len(r.events) == 0]
print(f"{len(zero_event_records)} zero-event records in the train split")

solo_batch = PragmaCollator()(zero_event_records[:1])
solo_batch.validate()
print(f"n_events: {solo_batch.n_events}, n_profile_tokens: {solo_batch.n_profile_tokens}")
assert solo_batch.n_events == 0
assert solo_batch.n_profile_tokens > 0

14 zero-event records in the train split
n_events: 0, n_profile_tokens: 10


## `event_mlm_labels` are all `-100` (no masking yet)

The field exists in the batch contract now so Phase 4 doesn't need to change `PragmaBatch`'s shape — only how this tensor gets filled.

In [7]:
all_ignored = bool((batch.event_mlm_labels == IGNORE_INDEX).all())
print(f"all event_mlm_labels == IGNORE_INDEX: {all_ignored}")
assert all_ignored

all event_mlm_labels == IGNORE_INDEX: True
